# KGE

The notebook is to help test KGE score. There will be an separate kge test.py for actual testing and tutorial for KGE.

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from scores.continuous import mse
from scores.continuous.correlation import pearsonr
from scores.continuous.standard_impl import kge

import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import xarray as xr

np.random.seed(42)  # Ensures consistent values across notebook runs

In [4]:
# Uncomment the line below to view detailed help information on the arguments to the correlation function
# help(kge)

In [5]:
def compute_kge_vrf_scores(sim,obs, weights=[1, 1, 1]):
    """
    Calculate the Kling-Gupta Efficiency (KGE) between observed and simulated data.
    References:
    Gupta, H. V., H. Kling, K. K. Yilmaz, and G. F. Martinez (2009), 
    Decomposition of the mean squared error and NSE performance criteria:
    Implications for improving hydrological modelling, Journal of Hydrology,
    377, 80–91, doi: 10.1016/j.jhydrol.2009.08.003.
    
    Parameters:
    obs : array_like
        Vector of observations
    sim : array_like
        Vector of simulations
    weights : array_like, optional
        A 3-element vector describing the weights for each term in KGE.
        defined by: weights = [s_r s_alpha s_beta] (see
                  equation 10 in Gupta et al. for definitions of s_r, 
                  s_alpha and s_beta)
        Default is [1, 1, 1].

    Returns:
    KGE_s : float
        Kling-Gupta Efficiency
        
    Notes
    Stats are calculated only from values for which both observations and
    simulations are not null values. NB: null values in the simulation
    series are permitted.
    
    """
    if len(weights) != 3:
        raise ValueError("Weights must be a 3-element vector")

    s_r, s_alpha, s_beta = weights

    if not isinstance(obs, np.ndarray):
        obs = np.array(obs)
    if not isinstance(sim, np.ndarray):
        sim = np.array(sim)

    if obs.ndim != 1 or sim.ndim != 1:
        raise ValueError("Inputs must be vectors")

    nanindx = ~np.isnan(obs) & ~np.isnan(sim)
    obs = obs[nanindx]
    sim = sim[nanindx]

    if np.any(np.isnan(sim)):
        print("Warning: Null values detected in sim vector")

    r = np.corrcoef(obs, sim)[0, 1]
    sigma_sim = np.std(sim)
    sigma_obs = np.std(obs)
    mu_sim = np.mean(sim)
    mu_obs = np.mean(obs)

    alpha = sigma_sim / sigma_obs
    beta = mu_sim / mu_obs

    ED_s = np.sqrt((s_r * (r - 1))**2 + (s_alpha * (alpha - 1))**2 + (s_beta * (beta - 1))**2)
    KGE_s = 1 - ED_s
    KGE = {}
    
    KGE['r'] = r
    KGE['alpha'] = alpha
    KGE['beta'] = beta
    KGE['kge'] = KGE_s
    

    return KGE

In [6]:
def compute_kge_hydroeval(simulations, evaluation,weights=[1, 1, 1]):
    """Original Kling-Gupta Efficiency (KGE) and its three components
    (r, α, β) as per `Gupta et al., 2009
    <https://doi.org/10.1016/j.jhydrol.2009.08.003>`_.

    Note, all four values KGE, r, α, β are returned, in this order.

    :Calculation Details:
        .. math::
           E_{\\text{KGE}} = 1 - \\sqrt{[r - 1]^2 + [\\alpha - 1]^2
           + [\\beta - 1]^2}
        .. math::
           r = \\frac{\\text{cov}(e, s)}{\\sigma({e}) \\cdot \\sigma(s)}
        .. math::
           \\alpha = \\frac{\\sigma(s)}{\\sigma(e)}
        .. math::
           \\beta = \\frac{\\mu(s)}{\\mu(e)}

        where *e* is the *evaluation* series, *s* is (one of) the
        *simulations* series, *cov* is the covariance, *σ* is the
        standard deviation, and *μ* is the arithmetic mean.

    """
    # calculate error in timing and dynamics r
    # (Pearson's correlation coefficient)
    s_r, s_alpha, s_beta = weights
    nanindx = ~np.isnan(evaluation) & ~np.isnan(simulations)
    evaluation = evaluation[nanindx]
    simulations = simulations[nanindx]

    sim_mean = np.mean(simulations, axis=0, dtype=np.float64)
    obs_mean = np.mean(evaluation, dtype=np.float64)

    r_num = np.sum((simulations - sim_mean) * (evaluation - obs_mean),
                   axis=0, dtype=np.float64)
    r_den = np.sqrt(np.sum((simulations - sim_mean) ** 2,
                           axis=0, dtype=np.float64)
                    * np.sum((evaluation - obs_mean) ** 2,
                             dtype=np.float64))
    r = r_num / r_den
    # calculate error in spread of flow alpha
    alpha = np.std(simulations, axis=0) / np.std(evaluation, dtype=np.float64)
    # calculate error in volume beta (bias of mean discharge)
    beta = (np.sum(simulations, axis=0, dtype=np.float64)
            / np.sum(evaluation, dtype=np.float64))
    # calculate the Kling-Gupta Efficiency KGE
    kge_ = 1 - np.sqrt((s_r*(r - 1)) ** 2 + (s_alpha*(alpha - 1)) ** 2 + (s_beta*(beta - 1)) ** 2)

    return np.vstack((kge_, r, alpha, beta))

# Check using correlation test data

In [7]:
DA1_CORR = xr.DataArray(
    np.array([[1, 2, 3], [0, 1, 0], [0.5, -0.5, 0.5], [3, 6, 3]]),
    dims=("space", "time"),
    coords=[
        ("space", ["w", "x", "y", "z"]),
        ("time", [1, 2, 3]),
    ],
)

DA2_CORR = xr.DataArray(
    np.array([[2, 4, 6], [6, 5, 6], [3, 4, 5], [3, np.nan, 3]]),
    dims=("space", "time"),
    coords=[
        ("space", ["w", "x", "y", "z"]),
        ("time", [1, 2, 3]),
    ],
)

DA3_CORR = xr.DataArray(
    np.array([[1, 2, 3], [3, 2.5, 3], [1.5, 2, 2.5], [1.5, np.nan, 1.5]]),
    dims=("space", "time"),
    coords=[
        ("space", ["w", "x", "y", "z"]),
        ("time", [1, 2, 3]),
    ],
)
DA4_CORR = xr.DataArray(
    np.array([[1, 3, 7], [2, 2, 8], [3, 1, 7]]),
    dims=("space", "time"),
    coords=[
        ("space", ["x", "y", "z"]),
        ("time", [1, 2, 3]),
    ],
)
DA5_CORR = xr.DataArray(
    np.array([1, 2, 3]),
    dims=("space"),
    coords=[("space", ["x", "y", "z"])],
)

EXP_CORR_KEEP_SPACE_DIM = xr.DataArray(
    np.array([1.0, -1.0, 0.0, np.nan]),
    dims=("space"),
    coords=[("space", ["w", "x", "y", "z"])],
)

EXP_CORR_REDUCE_ALL = xr.DataArray(1.0)

EXP_CORR_DIFF_SIZE = xr.DataArray(
    np.array([1.0, -1.0, 0.0]),
    dims=("time"),
    coords=[("time", [1, 2, 3])],
)

## Expected KGE values
EXP_KGE_KEEP_SPACE_DIM = xr.DataArray(
    np.array([0.2928932188134524, -1.2103875562418747, -0.44811448882050064, np.nan]),
    dims=("space"),
    coords=[("space", ["w", "x", "y", "z"])],
)
EXP_KGE_REDUCE_ALL = xr.DataArray(0.2928932188134524)

EXP_KGE_rho_returns_components = xr.DataArray(1.)
EXP_KGE_alpha_returns_components = xr.DataArray(0.5)
EXP_KGE_beta_returns_components = xr.DataArray(0.5)
 
EXP_KGE_returns_components = xr.Dataset({
            'kge': EXP_KGE_REDUCE_ALL,
            'rho': EXP_KGE_rho_returns_components,
            'alpha': EXP_KGE_alpha_returns_components,
            'beta': EXP_KGE_beta_returns_components,
        })   

EXP_KGE_Scaling_Factors = xr.DataArray(1 - np.sqrt((0.5*(1-1))**2 + (1.0*(0.5-1))**2 + (2*(0.5-1))**2)) 

EXP_KGE_DIFF_SIZE = xr.DataArray(
    np.array([1.0, -1.0, -1.8791915368841288]),
    dims=("time"),
    coords=[("time", [1, 2, 3])],
)

## Test with dataset
#
#        (DS_BIAS1, DS_BIAS2, None, "space", None, EXP_DS_PBIAS1),

DS1_CORR = xr.Dataset({"a": DA1_CORR, "b": DA2_CORR})
DS2_CORR = xr.Dataset({"a": DA2_CORR, "b": DA1_CORR})

EXP_DS_CORR = xr.Dataset({"a": EXP_CORR_KEEP_SPACE_DIM, "b": EXP_CORR_KEEP_SPACE_DIM})



In [8]:
# Check reduce dim arg
#        (DA1_CORR, DA2_CORR, None, "space", EXP_CORR_KEEP_SPACE_DIM),
 #       # Check preserve dim arg
#        (DA1_CORR, DA2_CORR, "time", None, EXP_CORR_KEEP_SPACE_DIM),
        # Check reduce all
#        (DA3_CORR, DA2_CORR, None, None, EXP_CORR_REDUCE_ALL),
 #       # Check different size arrays as input
#        (DA4_CORR, DA5_CORR, "space", None, EXP_CORR_DIFF_SIZE),

### 1. Check preserve dim arg

In [9]:
kge_hydroeval = []
kge_vrf_scores = []
for i in range(4):
    fcst = DA1_CORR.isel(space=i)
    obs = DA2_CORR.isel(space=i)        
    kge_hydroeval.append(compute_kge_hydroeval(fcst,obs).flatten())    
    val = compute_kge_vrf_scores(fcst,obs)    
    kge_vrf_scores.append([val['kge'], val['r'], val['alpha'], val['beta']])    
kge_hydroeval = np.vstack(kge_hydroeval)
kge_vrf_scores = np.vstack(kge_vrf_scores)
print(f"kge_hydroeval\n: {kge_hydroeval.T}")
print(f"kge_vrf_scores\n: {kge_vrf_scores.T}")
# score.continous.kge
kge_out = kge(DA1_CORR, DA2_CORR, reduce_dims=None, preserve_dims='space',return_components=True)
r = pearsonr(DA1_CORR, DA2_CORR, reduce_dims=None, preserve_dims='space')
print(f"pearsonr_r: {r}")
print(f"kge_scores: {kge_out}")
assert kge_out['kge'].equals(EXP_KGE_KEEP_SPACE_DIM), "kge_out does not match EXP_KGE_KEEP_SPACE_DIM"


kge_hydroeval
: [[ 0.29289322 -1.21038756 -0.44811449         nan]
 [ 1.         -1.          0.                 nan]
 [ 0.5         1.          0.57735027         nan]
 [ 0.5         0.05882353  0.04166667  1.        ]]
kge_vrf_scores
: [[ 0.29289322 -1.21038756 -0.44811449         nan]
 [ 1.         -1.          0.                 nan]
 [ 0.5         1.          0.57735027         nan]
 [ 0.5         0.05882353  0.04166667  1.        ]]
pearsonr_r: <xarray.DataArray (space: 4)> Size: 32B
array([ 1., -1.,  0., nan])
Coordinates:
  * space    (space) <U1 16B 'w' 'x' 'y' 'z'
kge_scores: <xarray.Dataset> Size: 144B
Dimensions:  (space: 4)
Coordinates:
  * space    (space) <U1 16B 'w' 'x' 'y' 'z'
Data variables:
    kge      (space) float64 32B 0.2929 -1.21 -0.4481 nan
    rho      (space) float64 32B 1.0 -1.0 0.0 nan
    alpha    (space) float64 32B 0.5 1.0 0.5774 nan
    beta     (space) float64 32B 0.5 0.05882 0.04167 1.0


c:\Users\shr015\Miniconda3\envs\scores\Lib\site-packages\numpy\lib\_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\shr015\AppData\Local\Temp\ipykernel_24364\3928647582.py:58: RuntimeWarning: invalid value encountered in scalar divide
  alpha = sigma_sim / sigma_obs


## 2. Check reduce dim arg

In [10]:
kge_hydroeval = []
kge_vrf_scores = []
for i in range(4):
    fcst = DA1_CORR.isel(space=i)
    obs = DA2_CORR.isel(space=i)        
    kge_hydroeval.append(compute_kge_hydroeval(fcst,obs).flatten())    
    val = compute_kge_vrf_scores(fcst,obs)    
    kge_vrf_scores.append([val['kge'], val['r'], val['alpha'], val['beta']])    
kge_hydroeval = np.vstack(kge_hydroeval)
kge_vrf_scores = np.vstack(kge_vrf_scores)
print(f"kge_hydroeval\n: {kge_hydroeval.T}")
print(f"kge_vrf_scores\n: {kge_vrf_scores.T}")
# score.continous.kge
kge_out = kge(DA1_CORR, DA2_CORR, reduce_dims="time",return_components=True)
r = pearsonr(DA1_CORR, DA2_CORR, reduce_dims="time")
print(f"pearsonr_r: {r}")
print(f"kge_scores: {kge_out}")
assert kge_out['kge'].equals(EXP_KGE_KEEP_SPACE_DIM), "kge_out does not match EXP_KGE_KEEP_SPACE_DIM"

kge_hydroeval
: [[ 0.29289322 -1.21038756 -0.44811449         nan]
 [ 1.         -1.          0.                 nan]
 [ 0.5         1.          0.57735027         nan]
 [ 0.5         0.05882353  0.04166667  1.        ]]
kge_vrf_scores
: [[ 0.29289322 -1.21038756 -0.44811449         nan]
 [ 1.         -1.          0.                 nan]
 [ 0.5         1.          0.57735027         nan]
 [ 0.5         0.05882353  0.04166667  1.        ]]
pearsonr_r: <xarray.DataArray (space: 4)> Size: 32B
array([ 1., -1.,  0., nan])
Coordinates:
  * space    (space) <U1 16B 'w' 'x' 'y' 'z'
kge_scores: <xarray.Dataset> Size: 144B
Dimensions:  (space: 4)
Coordinates:
  * space    (space) <U1 16B 'w' 'x' 'y' 'z'
Data variables:
    kge      (space) float64 32B 0.2929 -1.21 -0.4481 nan
    rho      (space) float64 32B 1.0 -1.0 0.0 nan
    alpha    (space) float64 32B 0.5 1.0 0.5774 nan
    beta     (space) float64 32B 0.5 0.05882 0.04167 1.0


c:\Users\shr015\Miniconda3\envs\scores\Lib\site-packages\numpy\lib\_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\shr015\AppData\Local\Temp\ipykernel_24364\3928647582.py:58: RuntimeWarning: invalid value encountered in scalar divide
  alpha = sigma_sim / sigma_obs


## 3. Reduce all dims

In [13]:
fcst = []
obs = []
for i in range(4):
    fcst.append(DA3_CORR.isel(space=i).values)
    obs.append(DA2_CORR.isel(space=i).values)   

fcst = np.concatenate(fcst) 
obs = np.concatenate(obs) 
kge_hydroeval = compute_kge_hydroeval(fcst,obs)
val = compute_kge_vrf_scores(fcst,obs)
kge_vrf_scores = [val['kge'], val['r'], val['alpha'], val['beta']] 
print(f"kge_hydroeval\n: {kge_hydroeval.T}")
print(f"kge_vrf_scores\n: {kge_vrf_scores}")
# score.continous.kge
kge_out = kge(DA3_CORR, DA2_CORR,return_components=True)
r = pearsonr(DA3_CORR, DA2_CORR)
print(f"pearsonr_r: {r}")
print(f"kge_scores: {kge_out}")
assert kge_out['kge'].equals(EXP_KGE_REDUCE_ALL), "kge_out does not match EXP_KGE_REDUCE_ALL"

kge_hydroeval
: [[0.29289322 1.         0.5        0.5       ]]
kge_vrf_scores
: [np.float64(0.2928932188134524), np.float64(1.0), np.float64(0.5), np.float64(0.5)]
pearsonr_r: <xarray.DataArray ()> Size: 8B
array(1.)
kge_scores: <xarray.Dataset> Size: 32B
Dimensions:  ()
Data variables:
    kge      float64 8B 0.2929
    rho      float64 8B 1.0
    alpha    float64 8B 0.5
    beta     float64 8B 0.5


## 4 Test with returns components

In [20]:
fcst = []
obs = []
for i in range(4):
    fcst.append(DA3_CORR.isel(space=i).values)
    obs.append(DA2_CORR.isel(space=i).values)   

fcst = np.concatenate(fcst) 
obs = np.concatenate(obs) 
kge_hydroeval = compute_kge_hydroeval(fcst,obs)
val = compute_kge_vrf_scores(fcst,obs)
kge_vrf_scores = [val['kge'], val['r'], val['alpha'], val['beta']] 
print(f"kge_hydroeval\n: {kge_hydroeval.T}")
print(f"kge_vrf_scores\n: {kge_vrf_scores}")
# score.continous.kge
kge_out = kge(DA3_CORR, DA2_CORR,return_components=True)
r = pearsonr(DA3_CORR, DA2_CORR)
print(f"pearsonr_r: {r}")
print(f"kge_scores: {kge_out}")
#assert kge_out.equals(EXP_KGE_returns_components), "kge_out does not match EXP_KGE_returns_components"
xr.testing.assert_allclose(kge_out, EXP_KGE_returns_components, rtol=1e-10, atol=1e-10), "rho component does not match EXP_KGE_returns_components"

kge_hydroeval
: [[0.29289322 1.         0.5        0.5       ]]
kge_vrf_scores
: [np.float64(0.2928932188134524), np.float64(1.0), np.float64(0.5), np.float64(0.5)]
pearsonr_r: <xarray.DataArray ()> Size: 8B
array(1.)
kge_scores: <xarray.Dataset> Size: 32B
Dimensions:  ()
Data variables:
    kge      float64 8B 0.2929
    rho      float64 8B 1.0
    alpha    float64 8B 0.5
    beta     float64 8B 0.5


(None, 'rho component does not match EXP_KGE_returns_components')

## 5 Test with scaling factors

In [23]:
## 4 Test with returns components
fcst = []
obs = []
for i in range(4):
    fcst.append(DA3_CORR.isel(space=i).values)
    obs.append(DA2_CORR.isel(space=i).values)   

fcst = np.concatenate(fcst) 
obs = np.concatenate(obs) 
kge_hydroeval = compute_kge_hydroeval(fcst,obs,weights=[0.5,1.0,2.0])
val = compute_kge_vrf_scores(fcst,obs,weights=[0.5,1.0,2.0])
kge_vrf_scores = [val['kge'], val['r'], val['alpha'], val['beta']] 
print(f"kge_hydroeval\n: {kge_hydroeval.T}")
print(f"kge_vrf_scores\n: {kge_vrf_scores}")
# score.continous.kge
kge_out = kge(DA3_CORR, DA2_CORR,scaling_factors=[0.5,1.0,2.0])
print(f"pearsonr_r: {r}")
print(f"kge_scores: {kge_out}")
assert kge_out.equals(EXP_KGE_Scaling_Factors), "kge_out does not match EXP_KGE_Scaling_Factors"

kge_hydroeval
: [[-0.11803399  1.          0.5         0.5       ]]
kge_vrf_scores
: [np.float64(-0.1180339887498949), np.float64(1.0), np.float64(0.5), np.float64(0.5)]
pearsonr_r: <xarray.DataArray ()> Size: 8B
array(1.)
kge_scores: <xarray.DataArray ()> Size: 8B
array(-0.11803399)


## 3.  Check different size arrays as input
 (DA4_CORR, DA5_CORR, "space", None, EXP_CORR_DIFF_SIZE),

In [27]:
kge_hydroeval = []
kge_vrf_scores = []
for i in range(3):
    fcst = DA4_CORR.isel(time=i)
    obs = DA5_CORR
    #val1 = compute_kge_hydroeval(fcst,obs)
    kge_hydroeval.append(compute_kge_hydroeval(fcst,obs).flatten())    
    val = compute_kge_vrf_scores(fcst,obs)
    kge_vrf_scores.append([val['kge'], val['r'], val['alpha'], val['beta']]) 
kge_hydroeval = np.vstack(kge_hydroeval)
kge_vrf_scores = np.vstack(kge_vrf_scores)
print(f"kge_hydroeval\n: {kge_hydroeval.T}")
print(f"kge_vrf_scores\n: {kge_vrf_scores.T}")
# score.continous.kge
kge_out = kge(DA4_CORR, DA5_CORR,reduce_dims='space',return_components=True)
r = pearsonr(DA4_CORR, DA5_CORR,reduce_dims='space')
print(f"pearsonr_r: {r}")
print(f"kge_scores: {kge_out}")
# xr.testing.assert_equal(result, expected)
# Use assert to check if they are equal
assert kge_out['kge'].equals(EXP_KGE_DIFF_SIZE), "kge_out does not match EXP_KGE_DIFF_SIZE"
#assert kge_out.equals(EXP_KGE_REDUCE_ALL), "kge_out does not match EXP_KGE_DIFF_SIZE"


kge_hydroeval
: [[ 1.         -1.         -1.87919154]
 [ 1.         -1.          0.        ]
 [ 1.          1.          0.57735027]
 [ 1.          1.          3.66666667]]
kge_vrf_scores
: [[ 1.         -1.         -1.87919154]
 [ 1.         -1.          0.        ]
 [ 1.          1.          0.57735027]
 [ 1.          1.          3.66666667]]
pearsonr_r: <xarray.DataArray (time: 3)> Size: 24B
array([ 1., -1.,  0.])
Coordinates:
  * time     (time) int64 24B 1 2 3
kge_scores: <xarray.Dataset> Size: 120B
Dimensions:  (time: 3)
Coordinates:
  * time     (time) int64 24B 1 2 3
Data variables:
    kge      (time) float64 24B 1.0 -1.0 -1.879
    rho      (time) float64 24B 1.0 -1.0 0.0
    alpha    (time) float64 24B 1.0 1.0 0.5774
    beta     (time) float64 24B 1.0 1.0 3.667


## 4. Test with dataset
Note that KGE score cannot handle xrrray.Dataset

## 5. Test with Dask

In [28]:
import dask
import dask.array

In [29]:
fcst = DA3_CORR.chunk()
obs = DA2_CORR.chunk()
result = kge(fcst, obs)
assert isinstance(result.data, dask.array.Array)
result = result.compute()
assert isinstance(result.data, np.ndarray)
xr.testing.assert_allclose(result, EXP_KGE_REDUCE_ALL)

## Further test with synthetic dataset

In [3]:
obs = 10 * np.random.random((50, 50))  # Generate obs sample with mean of about 5.0
obs = xr.DataArray(
    data=obs, 
    dims=["time", "x"],
    coords={"time": pd.date_range("2023-01-01", "2023-02-19"), "x": np.arange(0, 50)}
)

# Create forecasts
fcst1 = obs - 4      # Generate Forecast system 1 with with mean of about 1.0

fcst2 = 4.5 + np.random.random((50, 50))  # Generate Forecast system 2 with with mean of about 5.0
fcst2 = xr.DataArray(
    data=fcst2, 
    dims=["time", "x"],
    coords={"time": pd.date_range("2023-01-01", "2023-02-19"), "x": np.arange(0, 50)}
)

In [4]:
kge(fcst1, obs, reduce_dims ='x')

<xarray.DataArray (time: 50)> Size: 400B
array([0.10298597, 0.19100001, 0.16370464, 0.22684925, 0.22488271,
       0.22951234, 0.12488961, 0.23840378, 0.20973537, 0.23944553,
       0.32917008, 0.11956944, 0.05256907, 0.09292636, 0.23142055,
       0.24737197, 0.20336213, 0.02281454, 0.16416294, 0.13759389,
       0.25623959, 0.24364375, 0.27627421, 0.27960653, 0.23251516,
       0.00852524, 0.228448  , 0.23767233, 0.2001923 , 0.24965452,
       0.14212731, 0.20983596, 0.22688044, 0.21296412, 0.10995378,
       0.17193384, 0.20960552, 0.28097954, 0.18175669, 0.17174279,
       0.13943845, 0.2381995 , 0.27759332, 0.21507819, 0.21187726,
       0.11677142, 0.15567921, 0.20419274, 0.11963935, 0.27807697])
Coordinates:
  * time     (time) datetime64[ns] 400B 2023-01-01 2023-01-02 ... 2023-02-19

In [6]:
kge(fcst1, obs, preserve_dims = 'time')  # This and above should be same

<xarray.DataArray (time: 50)> Size: 400B
array([0.10298597, 0.19100001, 0.16370464, 0.22684925, 0.22488271,
       0.22951234, 0.12488961, 0.23840378, 0.20973537, 0.23944553,
       0.32917008, 0.11956944, 0.05256907, 0.09292636, 0.23142055,
       0.24737197, 0.20336213, 0.02281454, 0.16416294, 0.13759389,
       0.25623959, 0.24364375, 0.27627421, 0.27960653, 0.23251516,
       0.00852524, 0.228448  , 0.23767233, 0.2001923 , 0.24965452,
       0.14212731, 0.20983596, 0.22688044, 0.21296412, 0.10995378,
       0.17193384, 0.20960552, 0.28097954, 0.18175669, 0.17174279,
       0.13943845, 0.2381995 , 0.27759332, 0.21507819, 0.21187726,
       0.11677142, 0.15567921, 0.20419274, 0.11963935, 0.27807697])
Coordinates:
  * time     (time) datetime64[ns] 400B 2023-01-01 2023-01-02 ... 2023-02-19

In [13]:
assert kge(fcst1, obs, reduce_dims ='x').equals(kge(fcst1, obs, preserve_dims = 'time')) # type: ignore

In [7]:
kge(fcst1, obs, reduce_dims ='time')

<xarray.DataArray (x: 50)> Size: 400B
array([0.14824122, 0.21597968, 0.14135366, 0.28820335, 0.24982263,
       0.24368028, 0.19930624, 0.23127714, 0.12458266, 0.21278551,
       0.1419139 , 0.28121391, 0.24338907, 0.16295668, 0.22158745,
       0.24761381, 0.09049522, 0.22283181, 0.06483822, 0.16282207,
       0.26099376, 0.18127405, 0.21488195, 0.23598136, 0.27565212,
       0.25082019, 0.0637063 , 0.2940117 , 0.23945293, 0.18652521,
       0.16100964, 0.2559587 , 0.20006281, 0.15620923, 0.19926266,
       0.23428854, 0.16403145, 0.11254238, 0.1696494 , 0.13217798,
       0.16523353, 0.20908318, 0.2042216 , 0.17462303, 0.15797735,
       0.22654761, 0.13330655, 0.10233406, 0.30091478, 0.14554657])
Coordinates:
  * x        (x) int64 400B 0 1 2 3 4 5 6 7 8 9 ... 41 42 43 44 45 46 47 48 49

In [8]:
kge(fcst1, obs, preserve_dims ='x')  # This and above should be same

<xarray.DataArray (x: 50)> Size: 400B
array([0.14824122, 0.21597968, 0.14135366, 0.28820335, 0.24982263,
       0.24368028, 0.19930624, 0.23127714, 0.12458266, 0.21278551,
       0.1419139 , 0.28121391, 0.24338907, 0.16295668, 0.22158745,
       0.24761381, 0.09049522, 0.22283181, 0.06483822, 0.16282207,
       0.26099376, 0.18127405, 0.21488195, 0.23598136, 0.27565212,
       0.25082019, 0.0637063 , 0.2940117 , 0.23945293, 0.18652521,
       0.16100964, 0.2559587 , 0.20006281, 0.15620923, 0.19926266,
       0.23428854, 0.16403145, 0.11254238, 0.1696494 , 0.13217798,
       0.16523353, 0.20908318, 0.2042216 , 0.17462303, 0.15797735,
       0.22654761, 0.13330655, 0.10233406, 0.30091478, 0.14554657])
Coordinates:
  * x        (x) int64 400B 0 1 2 3 4 5 6 7 8 9 ... 41 42 43 44 45 46 47 48 49

In [14]:
assert kge(fcst1, obs, reduce_dims ='time').equals(kge(fcst1, obs, preserve_dims ='x')) # type: ignore

In [15]:
kge_s,kge_rho,kge_albpha,kge_beta = kge(fcst1, obs, preserve_dims='all',return_components=True)  # should returns nans for 

In [16]:
kge_s

<xarray.DataArray (time: 50, x: 50)> Size: 20kB
array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]])
Coordinates:
  * time     (time) datetime64[ns] 400B 2023-01-01 2023-01-02 ... 2023-02-19
  * x        (x) int64 400B 0 1 2 3 4 5 6 7 8 9 ... 41 42 43 44 45 46 47 48 49

In [17]:
kge_rho

<xarray.DataArray (time: 50, x: 50)> Size: 20kB
array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]])
Coordinates:
  * time     (time) datetime64[ns] 400B 2023-01-01 2023-01-02 ... 2023-02-19
  * x        (x) int64 400B 0 1 2 3 4 5 6 7 8 9 ... 41 42 43 44 45 46 47 48 49

In [18]:
kge_alpha

<xarray.DataArray (time: 50, x: 50)> Size: 20kB
array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]])
Coordinates:
  * time     (time) datetime64[ns] 400B 2023-01-01 2023-01-02 ... 2023-02-19
  * x        (x) int64 400B 0 1 2 3 4 5 6 7 8 9 ... 41 42 43 44 45 46 47 48 49

In [19]:
kge_beta

<xarray.DataArray (time: 50, x: 50)> Size: 20kB
array([[ -0.06797638,   0.57926372,   0.45354739, ...,   0.23086984,
          0.26835105,  -1.16386453],
       [  0.58745221,   0.48395941,   0.57424114, ...,   0.06441725,
        -14.73618181,  -2.7074308 ],
       [-11.72702398,   0.37147477,  -0.27244278, ...,   0.20426359,
         -6.77019625,  -0.43551077],
       ...,
       [  0.37116415,   0.56202155,   0.3470171 , ..., -14.71847587,
          0.38063466,   0.37196021],
       [ -0.17438375,  -4.57787567,   0.02350639, ...,  -1.05524671,
         -5.93871251,  -0.16806062],
       [ -0.48012365,   0.58610016,   0.28277986, ...,   0.53788059,
          0.59214443,   0.01860771]])
Coordinates:
  * time     (time) datetime64[ns] 400B 2023-01-01 2023-01-02 ... 2023-02-19
  * x        (x) int64 400B 0 1 2 3 4 5 6 7 8 9 ... 41 42 43 44 45 46 47 48 49